In [ ]:
# --- Xray Cloud: Fetch Accurate Dashboard Counts (Dynamic Reports API) ---

import requests, getpass, os, pandas as pd, datetime, urllib3, time

# ============================================================
# Setup
# ============================================================

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

CACHE_DIR = "./xray_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

XRAY_AUTH_URL   = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_REPORT_EXPORT_URL = "https://xray.cloud.getxray.app/api/v2/reports/export"
VERIFY_SSL = False  # Set to True in CI

# ============================================================
# Auth
# ============================================================

CID  = getpass.getpass("XRAY_CLIENT_ID: ")
CSEC = getpass.getpass("XRAY_CLIENT_SECRET: ")

def auth():
    r = requests.post(XRAY_AUTH_URL,
                      json={"client_id": CID, "client_secret": CSEC},
                      verify=VERIFY_SSL, timeout=30)
    r.raise_for_status()
    return r.text.strip('"')

TOKEN = auth()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# ============================================================
# Dynamic Report Fetch
# ============================================================

def fetch_test_plan_metrics(project_key: str, fix_version: str):
    """Generate the 'Test Plans Metrics' report dynamically via Reports API."""
    payload = {
        "reportName": "Test Plans Metrics",
        "params": {
            "projectKey": project_key,
            "fixVersion": fix_version,
            "maxResults": 2000
        },
        "format": "json"
    }

    print(f"⏳ Generating 'Test Plans Metrics' report for {project_key} / {fix_version} ...")
    r = requests.post(XRAY_REPORT_EXPORT_URL, headers=HEADERS, json=payload, verify=VERIFY_SSL, timeout=90)
    r.raise_for_status()

    data = r.json()
    rows = data.get("rows", [])
    if not rows:
        raise SystemExit("❌ No data returned from the report.")
    df = pd.DataFrame(rows)
    print(f"✅ Retrieved {len(df)} rows.")
    return df

# ============================================================
# Extract Dashboard Metrics
# ============================================================

def extract_plan_metrics(df: pd.DataFrame, plan_key: str):
    plan_key = plan_key.upper()
    match_cols = [c for c in df.columns if "key" in c.lower()]
    key_col = match_cols[0] if match_cols else None
    if not key_col:
        raise SystemExit("❌ Could not find a test plan key column in report output.")

    subset = df[df[key_col].str.upper() == plan_key]
    if subset.empty:
        raise SystemExit(f"❌ Plan {plan_key} not found in report output.")

    row = subset.iloc[0]
    total     = int(row.get("total", 0))
    passed    = int(row.get("passed", 0))
    failed    = int(row.get("failed", 0))
    descoped  = int(row.get("descoped", 0))
    blocked   = int(row.get("blocked", 0))
    executing = int(row.get("executing", 0))
    todo      = int(row.get("todo", 0))

    executed_final = passed + failed + descoped
    pass_rate      = round((passed / (passed + failed) * 100), 2) if (passed + failed) else 0
    coverage       = round((executed_final / total * 100), 2) if total else 0
    remaining      = round((todo / total * 100), 2) if total else 0

    summary = pd.DataFrame([{
        "Release": plan_key,
        "Planned Tests": total,
        "Passed": passed,
        "Failed": failed,
        "Descoped": descoped,
        "Blocked": blocked,
        "Executing": executing,
        "To-Do": todo,
        "Pass %": pass_rate,
        "Execution Coverage %": coverage,
        "Remaining %": remaining,
        "Timestamp": datetime.datetime.now().isoformat()
    }])
    return summary

# ============================================================
# Run
# ============================================================

PROJECT_KEY = "PROJ"        # Your Jira project
FIX_VERSION = "25.8"       # Release version
PLAN_KEY    = "PROJ-1001"  # Specific plan to extract

df_report = fetch_test_plan_metrics(PROJECT_KEY, FIX_VERSION)
df_dashboard = extract_plan_metrics(df_report, PLAN_KEY)

print("\n=== 📊 Xray Dashboard Metrics (Reports API) ===\n")
print(df_dashboard.to_string(index=False))

out_csv = os.path.join(CACHE_DIR, "xray_reports_dashboard.csv")
df_dashboard.to_csv(out_csv, index=False)
print(f"\n💾 Saved to {out_csv}")
